# Bronze Layer - Trip Ingestion

## Objective

Load raw NYC Taxi trip data and persist it as a Bronze Delta table.

### Input

- yellow_tripdata_2024-01.parquet

### Output

- workspace.default.bronze_trips

### Bronze Layer Principle

Preserve source data with minimal transformation while adding ingestion metadata.

In [0]:
# Read source trip data

trip_df = spark.read.parquet(
    "/Volumes/workspace/default/raw_data/yellow_tripdata_2024-01.parquet"
)

## Add Ingestion Metadata

Add basic lineage and audit information to support traceability.

In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    lit
)

In [0]:
bronze_trip_df = (
    trip_df
    .withColumn(
        "source_file",
        lit("yellow_tripdata_2024-01.parquet")
    )
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
)

## Validate Metadata

Verify the newly added metadata columns.

In [0]:
display(
    bronze_trip_df.select(
        "VendorID",
        "trip_distance",
        "source_file",
        "ingestion_timestamp"
    )
)

## Create Bronze Delta Table

Store the dataset in Delta format for reliable and efficient downstream processing.

In [0]:
bronze_trip_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.default.bronze_trips"
    )

## Validation

Confirm that all source records were successfully loaded into the Bronze table.

In [0]:
%sql
SELECT COUNT(*)
FROM workspace.default.bronze_trips

## Summary

| Output Table | Records |
|------------|----------|
| bronze_trips | 2,964,624 |

### Metadata Added

- source_file
- ingestion_timestamp

✅ Bronze Trip Ingestion Completed